# CARFAC SAI on drone search-and-rescue audio

Follow-on to [`carfac-vs-mel`](https://colab.research.google.com/github/6cubed/216labs/blob/main/colabs/carfac-vs-mel/experiment.ipynb): same two frontends, plus the **stabilized auditory image (SAI)**.

| Frontend | What it is |
|----------|------------|
| **Mel spectrogram** | STFT → mel filterbank → dB (librosa default path) |
| **CARFAC NAP** | Cascade of Asymmetric Resonators with Fast-Acting Compression → neural activity pattern ([google/carfac](https://github.com/google/carfac) NumPy) |
| **SAI** | Trigger-aligned running autocorrelation of the NAP — cochlear channel × **lag**, so periodicity shows up as vertical ridges ([`carfac.sai`](https://github.com/google/carfac/blob/master/python/src/carfac/sai.py)) |

Audio is **speech** and a **distress cry** from [DroneAudioSet](https://huggingface.co/datasets/ahlab-drone-project/DroneAudioSet) (drone-based search and rescue, [arXiv:2510.15383](https://arxiv.org/abs/2510.15383), MIT license) — each as the clean played-back source and as recorded by a mic array on a flying drone, where ego-noise buries the voice.

Not a bake-off — a visual sense of what survives in each representation.

**Paid detection pilots** (hydrophone, drone, bird): [6cubed.app/#work](https://6cubed.app/#work) · [CARFAC pilots](https://github.com/6cubed/216labs/blob/main/docs/CARFAC-PILOTS.md)


In [ ]:
# Install (Colab / fresh env). Safe to re-run.
%pip install -q "numpy" "matplotlib" "librosa" "soundfile" \
  "carfac @ git+https://github.com/google/carfac.git@master#subdirectory=python"


In [ ]:
from __future__ import annotations

import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
from IPython.display import Audio, display

from carfac.np import carfac as carfac_np
import carfac.sai as pysai

SR = 22050        # CARFAC NumPy default design rate
MAX_SECONDS = 3.0  # CARFAC NumPy is ~4x slower than real time; keep clips short

# Mel / NAP framing
N_FFT = 1024
HOP_LENGTH = 256
N_MELS = 96

# SAI framing: one SAI frame per SAI_SEG_WIDTH samples of NAP (~21 frames/s at 22.05 kHz)
SAI_SEG_WIDTH = 1024
SAI_WIDTH = 512      # lag columns
SAI_FUTURE_LAGS = 64  # columns to the right of the trigger (a little look-ahead)
SAI_TRIGGERS = 2      # trigger windows blended per frame
SAI_WARMUP_S = 0.5    # CARFAC's AGC needs a moment to settle; skip that when picking a frame


## 1. Audio — DroneAudioSet samples

The full dataset is 23.5 h of multi-channel recordings. We only need a few seconds, so we pull two sample
WAVs from the authors' code repo ([augmented-human-lab/DroneAudioSet-code](https://github.com/augmented-human-lab/DroneAudioSet-code)):

- `original-source-signals/File1.wav` — the clean signal played through a loudspeaker (mono, 44.1 kHz)
- `drone-with-source-recordings/mic3_8array-up-File1.wav` — the same session recorded by the drone's 8-mic
  array (16 kHz), room1 / drone1 / speaker at 1 m / mic at 25 cm

Both files run ~152 s through a fixed script: **male speech** (0–31 s), female speech (31–62 s),
**crying** (62–92 s), other human sounds (92–122 s), non-human sounds (122–152 s). We take one window from
the male-speech block and one from the crying block. Offsets are matched by that script, not sample-aligned.


In [ ]:
BASE = "https://raw.githubusercontent.com/augmented-human-lab/DroneAudioSet-code/main/audio-samples"
FILES = {
    "source": f"{BASE}/original-source-signals/File1.wav",
    "drone": f"{BASE}/drone-with-source-recordings/mic3_8array-up-File1.wav",
}
MIC_CHANNEL = 0  # which mic of the 8-element array to use (mono, no beamforming)

# (label, which file, offset in seconds)
CLIPS = [
    ("speech · clean source", "source", 16.0),
    ("speech · drone mic", "drone", 16.0),
    ("distress cry · clean source", "source", 70.0),
    ("distress cry · drone mic", "drone", 70.0),
]


def fetch(kind: str) -> Path:
    """Download a sample WAV once into the working directory."""
    path = Path(f"droneaudioset_{kind}.wav")
    if not path.exists():
        print(f"downloading {kind} ...")
        urllib.request.urlretrieve(FILES[kind], path)
    return path


def load_clip(kind: str, offset: float) -> np.ndarray:
    """Peak-normalized mono clip at SR. Multi-channel files use MIC_CHANNEL only."""
    y, _ = librosa.load(
        fetch(kind), sr=SR, mono=False, offset=offset, duration=MAX_SECONDS
    )
    if y.ndim > 1:
        y = y[MIC_CHANNEL]
    y = y.astype(np.float32)
    return y / (np.max(np.abs(y)) + 1e-9)


audio = {}
for label, kind, offset in CLIPS:
    audio[label] = load_clip(kind, offset)
    print(f"{label:30s} {len(audio[label])/SR:.2f}s @ {SR} Hz  (from {kind}, t={offset}s)")
    display(Audio(audio[label], rate=SR))


## 2. Frontends

`run_segment` returns the NAP with shape `(n_samples, n_channels)`. Mel and NAP are framed the same way so
their time axes line up. The SAI consumes the NAP in `SAI_SEG_WIDTH` chunks and returns one
`(n_channels, SAI_WIDTH)` frame per chunk.

Lag bookkeeping: `carfac.sai` puts the trigger (zero lag) at column `SAI_WIDTH - 1 - SAI_FUTURE_LAGS`, with
past lags to its left. We flip the frames so **lag increases to the right** — a voiced sound then shows a
ridge at its pitch period (and multiples of it).


In [ ]:
def mel_db(y: np.ndarray) -> np.ndarray:
    mel = librosa.feature.melspectrogram(
        y=y, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS, fmin=20.0, fmax=SR / 2
    )
    return librosa.power_to_db(mel, ref=np.max)


def carfac_nap(y: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return (n_samples, n_ch) NAP and pole center frequencies (Hz)."""
    cfp = carfac_np.design_carfac(n_ears=1, fs=float(SR))
    cfp = carfac_np.carfac_init(cfp)
    naps, cfp, _bm, _ohc, _agc = carfac_np.run_segment(cfp, y.astype(np.float64).reshape(-1, 1))
    if naps.ndim == 3:
        naps = naps[:, :, 0]
    pole_freqs = np.asarray(cfp.pole_freqs, dtype=np.float64).reshape(-1)
    return naps.astype(np.float32), pole_freqs


def frame_mean(x: np.ndarray, frame: int, hop: int) -> np.ndarray:
    """x: (n_samp, n_ch) -> (n_ch, n_frames) mean absolute activity per hop."""
    n_samp, _ = x.shape
    frames = []
    for start in range(0, max(n_samp - frame + 1, 1), hop):
        chunk = x[start : start + frame]
        if chunk.shape[0] < frame:
            break
        frames.append(np.mean(np.abs(chunk), axis=0))
    if not frames:
        frames = [np.mean(np.abs(x), axis=0)]
    return np.stack(frames, axis=1)


def nap_db(naps: np.ndarray) -> np.ndarray:
    framed = 20.0 * np.log10(frame_mean(naps, N_FFT, HOP_LENGTH) + 1e-8)
    return framed - np.max(framed)


def sai_frames(naps: np.ndarray) -> np.ndarray:
    """(n_samp, n_ch) NAP -> (n_frames, n_ch, SAI_WIDTH), lag increasing to the right."""
    n_samp, n_ch = naps.shape
    params = pysai.SAIParams(
        num_channels=n_ch,
        sai_width=SAI_WIDTH,
        future_lags=SAI_FUTURE_LAGS,
        num_triggers_per_frame=SAI_TRIGGERS,
        trigger_window_width=SAI_SEG_WIDTH,
        input_segment_width=SAI_SEG_WIDTH,
    )
    sai = pysai.SAI(params)
    nap_t = naps.T.astype(np.float64)  # SAI wants (n_ch, n_samp)
    frames = [
        sai.RunSegment(nap_t[:, s : s + SAI_SEG_WIDTH])[:, ::-1].copy()
        for s in range(0, n_samp - SAI_SEG_WIDTH + 1, SAI_SEG_WIDTH)
    ]
    return np.stack(frames, axis=0)


# Lag axis (ms) for the flipped frames: negative = future, 0 = trigger.
LAG_MS = (np.arange(SAI_WIDTH) - SAI_FUTURE_LAGS) * 1000.0 / SR
SAI_FRAME_RATE = SR / SAI_SEG_WIDTH


In [ ]:
import time

feat = {}
for label, _kind, _offset in CLIPS:
    t0 = time.time()
    y = audio[label]
    naps, pole_freqs = carfac_nap(y)
    feat[label] = {
        "y": y,
        "mel": mel_db(y),
        "nap": nap_db(naps),
        "sai": sai_frames(naps),
        "pole_freqs": pole_freqs,
    }
    print(f"{label:30s} sai={feat[label]['sai'].shape}  ({time.time() - t0:.1f}s)")

POLE_FREQS = feat[CLIPS[0][0]]["pole_freqs"]
print(
    f"\nCARFAC channels: {len(POLE_FREQS)}  CF range: {POLE_FREQS.min():.0f}-{POLE_FREQS.max():.0f} Hz"
    f"  |  SAI: {SAI_WIDTH} lags ({LAG_MS[0]:.1f} to {LAG_MS[-1]:.1f} ms) at {SAI_FRAME_RATE:.1f} frames/s"
)


## 3. Mel, NAP, and the SAI over time

The third column is the SAI summed over cochlear channels — a lag-vs-time view ("SAI-gram") of what the
frames below hold. Horizontal bands there are stable periodicities; the drone rows are dominated by
rotor-rate structure.


In [ ]:
def show_sai_gram(ax, sai: np.ndarray) -> None:
    gram = sai.sum(axis=1).T  # (lag, frames)
    gram = np.maximum(gram, 0.0) ** 0.5
    ax.imshow(
        gram,
        aspect="auto",
        origin="lower",
        cmap="magma",
        extent=[0, sai.shape[0] / SAI_FRAME_RATE, LAG_MS[0], LAG_MS[-1]],
        vmin=0.0,
        vmax=float(np.percentile(gram, 99.5)),
        interpolation="nearest",
    )
    ax.axhline(0.0, color="w", lw=0.6, ls=":", alpha=0.6)


fig, axes = plt.subplots(len(CLIPS), 3, figsize=(15, 3.2 * len(CLIPS)), constrained_layout=True)
for row, (label, _kind, _offset) in enumerate(CLIPS):
    f = feat[label]

    librosa.display.specshow(
        f["mel"], sr=SR, hop_length=HOP_LENGTH, x_axis="time", y_axis="mel",
        fmin=20.0, fmax=SR / 2, ax=axes[row, 0], cmap="magma",
    )
    axes[row, 0].set_title(f"{label} — mel (dB)", fontsize=10)

    axes[row, 1].imshow(
        f["nap"][::-1], aspect="auto", origin="lower", cmap="magma",
        extent=[0, len(f["y"]) / SR, float(POLE_FREQS.min()), float(POLE_FREQS.max())],
        interpolation="nearest",
    )
    axes[row, 1].set_title("CARFAC NAP (dB re peak)", fontsize=10)
    axes[row, 1].set_ylabel("approx CF (Hz)")
    axes[row, 1].set_xlabel("time (s)")

    show_sai_gram(axes[row, 2], f["sai"])
    axes[row, 2].set_title("SAI summed over channels", fontsize=10)
    axes[row, 2].set_ylabel("lag (ms)")
    axes[row, 2].set_xlabel("time (s)")

plt.show()


## 4. Stabilized auditory image frames

One frame each (~46 ms of NAP, blended over 2 trigger windows), taken at the loudest SAI frame of each clip.
Rows are cochlear channels (low CF at the bottom), columns are lag. Vertical ridges at a fixed lag are
periodic energy; a voiced pitch of *f0* puts them at 1/*f0* and its multiples.


In [ ]:
def pick_frame(sai: np.ndarray) -> int:
    """Loudest SAI frame, ignoring the AGC settling transient at the start of the clip."""
    energy = sai.sum(axis=(1, 2))
    energy[: int(SAI_WARMUP_S * SAI_FRAME_RATE)] = -np.inf
    return int(np.argmax(energy))


def show_sai_frame(ax, sai: np.ndarray, title: str) -> None:
    idx = pick_frame(sai)
    frame = np.maximum(sai[idx], 0.0) ** 0.5
    ax.imshow(
        frame[::-1],
        aspect="auto",
        origin="lower",
        cmap="magma",
        extent=[LAG_MS[0], LAG_MS[-1], float(POLE_FREQS.min()), float(POLE_FREQS.max())],
        vmin=0.0,
        vmax=float(np.percentile(frame, 99.5)),
        interpolation="nearest",
    )
    ax.axvline(0.0, color="w", lw=0.8, ls=":", alpha=0.7)
    ax.set_title(f"{title}  (frame {idx}, t≈{idx / SAI_FRAME_RATE:.2f}s)", fontsize=10)
    ax.set_xlabel("lag (ms)")
    ax.set_ylabel("approx CF (Hz)")


fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
for ax, (label, _kind, _offset) in zip(axes.ravel(), CLIPS):
    show_sai_frame(ax, feat[label]["sai"], label)
plt.show()

print(
    "Read tip: the clean rows should show a comb of vertical ridges at the voice pitch period "
    "(roughly 4-10 ms for adult speech, shorter for a cry). On the drone rows the same lags are "
    "competing with rotor periodicity and broadband ego-noise at SNRs well below 0 dB."
)


## Try other clips

- Change the offsets in `CLIPS` to land in another block of the 152 s script (female speech at 31–62 s,
  clapping and other human sounds at 92–122 s, non-human sounds at 122–152 s).
- Switch `MIC_CHANNEL` to hear/see a different element of the 8-mic array.
- Raise `SAI_WIDTH` for longer lags (lower pitches), or `SAI_TRIGGERS` for more blending per frame.
- Swap in the full dataset from Hugging Face (`ahlab-drone-project/DroneAudioSet`) for other rooms, drones,
  throttles, and mic placements — the parquet shards are ~120 MB each.
